In [ ]:
# Setup Colab dengan Google Drive (sama seperti notebook EN).
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
    PROJECT_DIR = "/content/drive/MyDrive/Aini/ml-assignment/Team-Assignment-2/binus-ai-2026sem3-assignment2-group04"
    MODEL_DIR   = f"{PROJECT_DIR}/models"
    FIGURES_DIR = f"{PROJECT_DIR}/report/figures"
    RESULTS_CSV = f"{PROJECT_DIR}/report/realworld_results.csv"
except ImportError:
    ON_COLAB = False
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
    os.chdir(PROJECT_ROOT)
    MODEL_DIR   = "models"
    FIGURES_DIR = "report/figures"
    RESULTS_CSV = "report/realworld_results.csv"

MODEL_PATH = f"{MODEL_DIR}/transfer_mobilenet_v1.keras"

# Locate test_images/ — sama dengan EN notebook
candidates = [
    "/content/test_images",
    f"{PROJECT_DIR}/test_images" if ON_COLAB else None,
    "/content/repo/test_images",
    "test_images",
]
TEST_IMAGES_DIR = next((c for c in candidates if c and Path(c).is_dir() and any(Path(c).iterdir())), None)

import tensorflow as tf
print("On Colab:    ", ON_COLAB)
print("TensorFlow:  ", tf.__version__)
print("CSV path:    ", RESULTS_CSV)
print("Test images: ", TEST_IMAGES_DIR)

# Tugas Kelompok 2 — Phase 3A: Pengujian Gambar Dunia Nyata

**Tujuan:**
Mengevaluasi model transfer learning yang sudah terlatih (`transfer_mobilenet_v1.keras`, akurasi tes 0,8993 pada test set CIFAR-10) pada **30 gambar eksternal** yang bersumber dari web terbuka. Test split CIFAR-10 adalah in-distribution secara konstruksi; notebook ini mengkuantifikasi bagaimana model berperilaku pada foto dunia nyata yang out-of-distribution dan mengidentifikasi mode kegagalan sistematis untuk diskusi Bab 8 laporan.

**Metodologi:**
- 30 gambar, 3 per kelas, terorganisir sebagai `test_images/<kelas>/<kelas>_NN.jpg`.
- Sumber: Unsplash, Pexels, hasil pencarian gratis Google Images. Dikecualikan: gambar asli CIFAR-10 dan gambar yang dihasilkan AI.
- Pipeline preprocessing yang sama dengan inferensi (`center_crop_square → resize 96 → mobilenet_v2.preprocess_input`) sehingga notebook ini mencerminkan apa yang dilakukan aplikasi Streamlit yang di-deploy.

**Catatan untuk versi ID:** versi ini menerapkan **option C** — daripada menjalankan inferensi ulang (yang memerlukan upload `test_images/` ke Colab lagi), notebook ini memuat `realworld_results.csv` yang sudah dihasilkan oleh `04_realworld_testing_EN.ipynb` dan menampilkannya kembali dengan label dan plot dalam Bahasa Indonesia. Hasil numerik identik karena bersumber dari run yang sama. Cabang fallback (else) tetap memuat model dan menjalankan inferensi jika CSV tidak ditemukan.

**Hasil yang diharapkan:**
Akurasi dunia nyata akan terlihat lebih rendah dari angka 0,8993 CIFAR-test. Selisihnya mengkuantifikasi pergeseran domain antara thumbnail 32×32 dan foto resolusi tinggi alami, yang merupakan keterbatasan utama yang ditandai di seluruh laporan.

### Bagian 0: Setup dan Reproduktibilitas

Notebook ini hanya melakukan inferensi (atau memuat hasil cache) — tidak ada training, tidak ada randomness pada modeling. Hasil bersifat bit-deterministik diberikan gambar input yang sama dan artifact `transfer_mobilenet_v1.keras` yang sama.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Class names — versi Indonesia (untuk display)
CLASS_NAMES_EN = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                  'dog', 'frog', 'horse', 'ship', 'truck']
CLASS_NAMES_ID = ['pesawat', 'mobil', 'burung', 'kucing', 'rusa',
                  'anjing', 'katak', 'kuda', 'kapal', 'truk']
EN_TO_ID = dict(zip(CLASS_NAMES_EN, CLASS_NAMES_ID))
ID_TO_EN = dict(zip(CLASS_NAMES_ID, CLASS_NAMES_EN))

NUM_CLASSES = 10
IMG_SIZE = 96

print(f"Kelas (ID): {CLASS_NAMES_ID}")
print(f"Ukuran gambar: {IMG_SIZE}x{IMG_SIZE}")

### Bagian 1: Memuat Hasil dari Run EN

Cabang utama memuat `realworld_results.csv` yang sudah dihasilkan dari run EN. Cabang fallback menjalankan inferensi penuh jika CSV tidak ditemukan.

In [ ]:
CSV_LOADED = os.path.exists(RESULTS_CSV)

class _StubInferenceResult:
    """Stub untuk menyimpan hasil inferensi yang dimuat dari CSV."""
    def __init__(self, df): self.df = df

if CSV_LOADED:
    print(f"Memuat hasil dari {RESULTS_CSV}")
    print("(Melewati inferensi ulang — hasil dihasilkan oleh 04_realworld_testing_EN.ipynb)")
    df = pd.read_csv(RESULTS_CSV)
    print(f"\nMemuat {len(df)} baris hasil.")
else:
    print("CSV tidak ditemukan, menjalankan inferensi penuh...")
    # Fallback: load model and run inference (sama seperti EN)
    assert TEST_IMAGES_DIR is not None, "test_images/ tidak ditemukan; tidak bisa menjalankan inferensi."
    assert os.path.exists(MODEL_PATH), f"Model tidak ditemukan di {MODEL_PATH}"

    def center_crop_square(arr):
        h, w = arr.shape[:2]
        s = min(h, w)
        y, x = (h - s) // 2, (w - s) // 2
        return arr[y:y + s, x:x + s]

    def load_and_preprocess(path):
        img = Image.open(path).convert("RGB")
        rgb = np.array(img)
        rgb = center_crop_square(rgb)
        x = tf.cast(rgb, tf.float32)
        x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))
        x = preprocess_input(x)
        return rgb, x.numpy()

    model = tf.keras.models.load_model(MODEL_PATH)
    _ = model(np.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32), training=False)

    rows = []
    for class_idx, class_name in enumerate(CLASS_NAMES_EN):
        for img_path in sorted((Path(TEST_IMAGES_DIR) / class_name).glob("*.jpg")):
            _, x = load_and_preprocess(str(img_path))
            proba = model(x[np.newaxis], training=False).numpy()[0]
            top3_idx = np.argsort(proba)[-3:][::-1]
            rows.append({
                "filename":   img_path.name,
                "true_class": class_name,
                "true_idx":   class_idx,
                "pred_class": CLASS_NAMES_EN[int(top3_idx[0])],
                "pred_idx":   int(top3_idx[0]),
                "confidence": float(proba[top3_idx[0]]),
                "top3":       "→".join(CLASS_NAMES_EN[int(i)] for i in top3_idx),
                "top3_conf":  "→".join(f"{proba[int(i)]:.2f}" for i in top3_idx),
                "correct":    bool(top3_idx[0] == class_idx),
            })
    df = pd.DataFrame(rows)
    print(f"\nMenghasilkan {len(df)} baris.")

### Bagian 2: Tabel Hasil Per Gambar

Tampilkan tabel hasil dengan kolom dalam Bahasa Indonesia.

In [ ]:
# Buat tabel display dengan label Indonesia
df_display = df.copy()
df_display["kelas_sebenarnya"] = df_display["true_class"].map(EN_TO_ID)
df_display["kelas_prediksi"]   = df_display["pred_class"].map(EN_TO_ID)
df_display["confidence"]       = df_display["confidence"].round(3)
df_display["benar"]            = df_display["correct"].map({True: "✓", False: "✗"})

# Susun ulang kolom untuk display
display_cols = ["filename", "kelas_sebenarnya", "kelas_prediksi", "confidence", "benar"]
df_display[display_cols]

### Bagian 3: Akurasi Agregat dan Per Kelas

Angka utama untuk laporan. Kami membandingkan akurasi dunia nyata langsung dengan angka CIFAR-test (0,8993) yang dicapai model pada split in-distribution.

In [ ]:
CIFAR_TEST_ACC = 0.8993       # dari 03_transfer_learning_EN.ipynb Bagian 8

overall_acc = df["correct"].mean()
print(f"Akurasi dunia nyata: {overall_acc:.4f}  ({df['correct'].sum()}/{len(df)} benar)")
print(f"Akurasi CIFAR test:  {CIFAR_TEST_ACC:.4f}")
print(f"Domain gap:          {(CIFAR_TEST_ACC - overall_acc) * 100:+.2f} pp\n")

per_class = (
    df.groupby("true_class")["correct"]
      .agg(["mean", "sum", "count"])
      .rename(columns={"mean": "akurasi", "sum": "n_benar", "count": "n_total"})
      .reindex(CLASS_NAMES_EN)
)
per_class.index = [EN_TO_ID[c] for c in per_class.index]
per_class.index.name = "kelas"

print("Akurasi per kelas (dunia nyata):")
print(per_class.to_string())

### Bagian 4: Visualisasi Kasus Kegagalan

Bangun grid yang menampilkan setiap gambar yang salah klasifikasi, dianotasi dengan kelas sebenarnya dan prediksi top model (dengan confidence). Ini adalah figure diagnostik untuk laporan — melihat gambar kegagalan aktual adalah satu-satunya cara untuk mengidentifikasi *jenis* domain shift apa yang melukai model.

In [ ]:
failures = df[~df["correct"]].reset_index(drop=True)
print(f"Kegagalan: {len(failures)} / {len(df)} ({len(failures) / len(df):.1%})\n")

if len(failures) == 0:
    print("Skor sempurna — tidak ada grid kegagalan untuk diplot.")
elif TEST_IMAGES_DIR is None:
    print("test_images/ tidak ditemukan — tidak bisa menampilkan grid (gambar tidak diunggah ke Colab).")
    print("Lihat figure tersimpan di report/figures/realworld-failures.png")
else:
    def center_crop_square(arr):
        h, w = arr.shape[:2]
        s = min(h, w)
        y, x = (h - s) // 2, (w - s) // 2
        return arr[y:y + s, x:x + s]

    n = len(failures)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for i, row in failures.iterrows():
        path = Path(TEST_IMAGES_DIR) / row["true_class"] / row["filename"]
        img = np.array(Image.open(path).convert("RGB"))
        img = center_crop_square(img)
        axes[i].imshow(img)
        true_id = EN_TO_ID[row["true_class"]]
        pred_id = EN_TO_ID[row["pred_class"]]
        axes[i].set_title(
            f"sebenarnya: {true_id}\nprediksi: {pred_id} ({row['confidence']:.0%})",
            fontsize=10,
        )

    plt.suptitle(
        f"Kegagalan dunia nyata — {len(failures)} dari {len(df)} ({len(failures) / len(df):.1%})",
        fontsize=13, y=1.0,
    )
    plt.tight_layout()
    plt.show()

### Bagian 5: Grafik Batang Akurasi Per Kelas

Bandingkan akurasi setiap kelas dengan plafon CIFAR-test (garis putus-putus).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
acc_vals = per_class["akurasi"].values
bars = ax.bar(per_class.index, acc_vals, color="steelblue", edgecolor="black")
ax.axhline(CIFAR_TEST_ACC, color="red", linestyle="--", label=f"Akurasi CIFAR test = {CIFAR_TEST_ACC:.3f}")
ax.set_ylabel("Akurasi dunia nyata")
ax.set_ylim(0, 1.05)
ax.set_title("Akurasi per kelas pada 30 gambar dunia nyata")
for bar, v in zip(bars, acc_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.2f}",
            ha="center", fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()

### Bagian 6: Diskusi

**Apa yang diukur ini.** Test set CIFAR-10 adalah bit-perfect in-distribution: sumber 32×32 yang sama, thumbnail bergaya kamera yang sama, kurasi penulis dataset yang sama. 30 gambar dunia nyata di sini sengaja *di luar* distribusi tersebut — foto alami resolusi tinggi yang bersumber dari perpustakaan foto publik, kemudian di-center-crop dan di-resize ke 96×96 oleh pipeline kami. Selisih akurasi antara angka CIFAR-test dan angka dunia nyata karenanya adalah pengukuran pergeseran domain alih-alih kelemahan model fundamental.

**Mode kegagalan yang harus dicari di grid di atas.** Tiga pola berulang pada model CIFAR-trained pada foto alami: (1) **ketidaksesuaian skala** — objek dunia nyata bisa memenuhi frame atau berada kecil di scene yang kompleks, sementara objek CIFAR selalu menempati sebagian besar frame 32×32; (2) **latar belakang mendominasi** — latar belakang alami (dedaunan, langit, air) membawa tekstur lebih kaya daripada thumbnail CIFAR, kadang menarik prediksi ke kelas yang distribusi training-nya berbagi latar belakang; (3) **kerancuan pasangan kelas** — kerancuan pasangan hewan yang sama yang terlihat di confusion matrix CIFAR (kucing/anjing, rusa/kuda, burung/pesawat) muncul kembali di sini, sering lebih jelas.

**Implikasi untuk app yang di-deploy.** Halaman Streamlit Predict sudah menyertakan disclaimer tentang domain gap. Pengguna yang mengunggah foto alami resolusi tinggi harus berharap confidence lebih rendah dan misclassification sesekali, terutama pada kelas pasangan hewan yang sulit. Augmentation yang lebih kuat atau backbone yang lebih besar dapat menutup sebagian gap ini, tetapi menghilangkannya sepenuhnya akan memerlukan fine-tuning pada gambar dunia nyata berlabel — pekerjaan yang ditandai sebagai "pekerjaan masa depan" di Bab 9 laporan.